# PaperTrail: 595-Paper Full-Text Evaluation

This notebook separates three tasks:
1. Run the **22 quick unit and integration tests**.
2. Visualize the retained **595-paper / 12,836-page evaluation** without downloading PDFs.
3. Optionally rebuild the full corpus and recompute every metric.

The full evaluation uses 595 successfully extracted papers, not 595 pages. Five of the original 600 records are documented exclusions.

## 1. Clone the project and install dependencies
Run the cells in order. The setup is safe to rerun because an existing clone is reused.

In [ ]:
import os

repo = '/content/NLP-project-research-paper-summarizer'
if not os.path.isdir(repo):
    !git clone https://github.com/notthedonrobbie/NLP-project-research-paper-summarizer.git /content/NLP-project-research-paper-summarizer
%cd /content/NLP-project-research-paper-summarizer

In [ ]:
!python -m pip install -r requirements.txt matplotlib

## 2. Run the quick test suite
These 22 tests check PDF extraction, page-aware chunking, retrieval, ROUGE, summarization, grounded answers, abstention, and the offline pipeline. They do not take the place of the 595-paper evaluation below.

In [ ]:
!python -m unittest discover -s tests -v

## 3. Inspect the retained full-text results
The repository includes aggregate metrics, per-paper ranks, generated summaries, and extraction diagnostics. No PDF download is needed for this section or the charts.

In [ ]:
import json
from pathlib import Path
import pandas as pd

result_path = Path('results/full_text/evaluation.json')
profile_path = Path('results/full_text/corpus_profile.json')
result = json.loads(result_path.read_text(encoding='utf-8'))
profile = json.loads(profile_path.read_text(encoding='utf-8'))

assert result['papers'] == 595
assert profile['pages']['total'] == 12836
assert len(result['retrieval']['per_query']) == 595
assert len(result['summarization']['per_paper']) == 595

corpus_table = pd.DataFrame({
    'Measure': ['Usable full papers', 'Extracted pages', 'Extracted words', 'Excluded records', 'Coverage below 0.80'],
    'Value': [
        profile['extracted_full_papers'],
        profile['pages']['total'],
        profile['extracted_words']['total'],
        len(profile['excluded_error_records']),
        profile['abstract_token_coverage']['below_0_80'],
    ],
})
display(corpus_table)

retrieval_table = pd.DataFrame({
    method.upper(): {
        'MRR': values['mrr'],
        'Hit@1': values['hit_rate']['k1']['mean'],
        'Hit@3': values['hit_rate']['k3']['mean'],
        'Hit@5': values['hit_rate']['k5']['mean'],
        'Hit@10': values['hit_rate']['k10']['mean'],
        'Top-10 misses': values['not_found_in_top10'],
    }
    for method, values in result['retrieval']['methods'].items()
}).T
display(retrieval_table.round(4))

summary_table = pd.DataFrame({
    ('LEAD' if method == 'lead' else 'TextRank'): values['macro_average']
    for method, values in result['summarization']['methods'].items()
}).T
summary_table.columns = ['ROUGE-1', 'ROUGE-2', 'ROUGE-L']
display(summary_table.round(4))

## 4. Visualize retrieval performance
Each title is a known-item query against all 595 full extracted papers. Printed titles inside PDFs can make this task optimistic; these scores are not natural-question answering accuracy.

In [ ]:
import matplotlib.pyplot as plt

cutoffs = [1, 3, 5, 10]
colors = {'bm25': '#526D9B', 'tfidf': '#2F665C'}
fig, ax = plt.subplots(figsize=(8, 4.8))
for method in ['bm25', 'tfidf']:
    values = [
        result['retrieval']['methods'][method]['hit_rate'][f'k{k}']['mean']
        for k in cutoffs
    ]
    ax.plot(cutoffs, values, marker='o', linewidth=2.5, markersize=7,
            label=method.upper(), color=colors[method])
    for k, value in zip(cutoffs, values):
        ax.annotate(f'{value:.3f}', (k, value), xytext=(0, 8),
                    textcoords='offset points', ha='center', fontsize=9)
ax.set(title='Known-item title-to-full-text retrieval',
       xlabel='Cut-off k', ylabel='Hit rate', xticks=cutoffs, ylim=(0.85, 1.01))
ax.grid(axis='y', alpha=0.25)
ax.legend()
fig.tight_layout()
fig.savefig('results/full_text/colab_retrieval.png', dpi=160)
plt.show()

## 5. Visualize extractive summarization
Six-sentence LEAD and TextRank extracts are compared with author abstracts using macro-average ROUGE F1. Abstracts and references are conservatively excluded from the summarization input. ROUGE measures lexical alignment, not factual faithfulness or human-rated usefulness.

In [ ]:
import numpy as np

metrics = [('rouge1', 'ROUGE-1'), ('rouge2', 'ROUGE-2'), ('rougeL', 'ROUGE-L')]
x = np.arange(len(metrics))
width = 0.34
fig, ax = plt.subplots(figsize=(8, 4.8))
for offset, method, label, color in [
    (-width / 2, 'lead', 'LEAD', '#C68A3C'),
    (width / 2, 'textrank', 'TextRank', '#7B658E'),
]:
    values = [
        result['summarization']['methods'][method]['macro_average'][metric]
        for metric, _ in metrics
    ]
    bars = ax.bar(x + offset, values, width, label=label, color=color)
    ax.bar_label(bars, fmt='%.3f', padding=3, fontsize=9)
ax.set(title='Six-sentence extractive summarization',
       ylabel='Macro-average ROUGE F1', xticks=x,
       xticklabels=[label for _, label in metrics], ylim=(0, 0.40))
ax.grid(axis='y', alpha=0.25)
ax.legend()
fig.tight_layout()
fig.savefig('results/full_text/colab_summarization.png', dpi=160)
plt.show()

## 6. Optional: rebuild and reevaluate all 595 papers
**This step is not needed for the charts above.** Set `REBUILD_FULL_CORPUS = True` only when you want a complete independent run.

- Downloads roughly 1.2 GiB of PDFs and processes 6.6 million extracted words.
- Can take substantial time and may require additional Colab memory.
- The downloader is resumable: rerun the cell after a timeout or disconnect.
- Four HTTP 404 downloads and one parser-limit failure are expected exclusions; do not use `--retry-errors` unless you intentionally want to retry them.
- Colab storage is temporary. Preserve `data/full_text/` separately if you need the bulk PDFs after the session.

After a successful rebuild, rerun Sections 3–5 to refresh the tables and charts.

In [ ]:
import subprocess
import sys

REBUILD_FULL_CORPUS = False

if REBUILD_FULL_CORPUS:
    subprocess.run([sys.executable, 'scripts/build_full_text_corpus_parallel.py'], check=True)
    text_files = list(Path('data/full_text/text').glob('paper-*.json'))
    assert len(text_files) == 595, f'Expected 595 extracted papers, found {len(text_files)}'
    subprocess.run([sys.executable, 'scripts/evaluate_full_text_corpus.py'], check=True)
    subprocess.run([sys.executable, 'scripts/validate_full_text_results.py'], check=True)
    result = json.loads(result_path.read_text(encoding='utf-8'))
    profile = json.loads(profile_path.read_text(encoding='utf-8'))
    print('Full-corpus evaluation and independent metric checks completed.')
else:
    print('Full rebuild skipped. The retained 595-paper results and visualizations remain available above.')